# TeleRAG: Full Ingestion Pipeline (Kaggle GPU)

This notebook runs the complete ingestion pipeline on Kaggle's T4 GPU:
1. Parse all 3GPP PDFs using PyMuPDF TOC-based extraction
2. Enrich metadata
3. Hierarchical chunking (leaf 200-500 tokens, section chunks)
4. Build Knowledge Graph
5. Embed all chunks with bge-large-en-v1.5 (GPU-accelerated)
6. Save outputs for download

**Prerequisites:** Upload your 15 3GPP PDFs as a Kaggle dataset.

In [ ]:
# Cell 1: Install dependencies
!pip install -q pymupdf sentence-transformers networkx

In [ ]:
# Cell 2: Configuration
import os

# ===== CHANGE THIS to your Kaggle dataset path =====
PDF_DIR = "/kaggle/input/3gpp-specs"  # Directory containing your PDFs
OUTPUT_DIR = "/kaggle/working/ingestion_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# List PDFs
pdf_files = sorted([f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')])
print(f"Found {len(pdf_files)} PDFs:")
for f in pdf_files:
    size_mb = os.path.getsize(os.path.join(PDF_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

In [ ]:
# Cell 3: PDF Parser (TOC-based, same as our local version)
import re
import fitz  # PyMuPDF
from typing import List, Dict, Any
from pathlib import Path

class TelecomPDFParser:
    """Parses 3GPP/ETSI PDFs using built-in TOC for reliable section detection."""

    def __init__(self):
        self.ts_ref_regex = re.compile(r"TS\s+(\d{2}\.\d{3})")
        self.clause_ref_regex = re.compile(r"clause\s+(\d+(?:\.\d+)*)")
        self._skip_titles = {
            "intellectual property rights", "legal notice",
            "modal verbs terminology", "foreword", "contents",
        }

    def _extract_spec_number(self, pdf_path):
        name = Path(pdf_path).stem.lower()
        match = re.search(r"ts_1(\d{2})(\d{3})", name)
        if match:
            return f"TS {match.group(1)}.{match.group(2)}"
        match2 = re.search(r"TS_(\d{2}\.\d{3})", Path(pdf_path).name, re.IGNORECASE)
        if match2:
            return f"TS {match2.group(1)}"
        return "Unknown"

    def _extract_release(self, pdf_path):
        match = re.search(r"v(\d{2})", Path(pdf_path).stem.lower())
        return match.group(1) if match else "18"

    def _extract_clause_string(self, title):
        match = re.match(r"^(\d+(?:\.\d+)*)\s+(.+)$", title.strip())
        if match:
            return match.group(1), match.group(2).strip()
        return "", title.strip()

    def parse(self, pdf_path):
        pdf_path = Path(pdf_path)
        sections = []
        try:
            doc = fitz.open(pdf_path)
        except Exception as e:
            print(f"  ERROR opening {pdf_path.name}: {e}")
            return []

        spec_number = self._extract_spec_number(pdf_path)
        release = self._extract_release(pdf_path)
        toc = doc.get_toc()
        total_pages = len(doc)

        if not toc:
            print(f"  WARNING: {pdf_path.name} has no TOC, using regex fallback")
            doc.close()
            return []

        # Build section list from TOC
        toc_entries = []
        for i, (level, title, start_page) in enumerate(toc):
            if title.strip().lower() in self._skip_titles:
                continue
            end_page = total_pages
            for j in range(i + 1, len(toc)):
                end_page = toc[j][2]
                break
            toc_entries.append({"level": level, "title": title, "start_page": start_page, "end_page": end_page})

        # Extract text for each section
        for entry in toc_entries:
            clause_str, clean_title = self._extract_clause_string(entry["title"])
            clause_path = clause_str.split(".") if clause_str else []
            start_pg = max(0, entry["start_page"] - 1)
            end_pg = min(total_pages, entry["end_page"] - 1)

            text_parts = []
            for pg in range(start_pg, end_pg + 1):
                page_text = doc[pg].get_text("text")
                lines = page_text.split("\n")
                cleaned = []
                for line in lines:
                    s = line.strip()
                    if s.startswith("ETSI TS ") or s.startswith("ETSI") or s.startswith("3GPP TS "):
                        continue
                    if re.match(r"^\d+$", s):
                        continue
                    cleaned.append(line)
                text_parts.append("\n".join(cleaned))

            content = "\n".join(text_parts).strip()
            if len(content) < 20:
                continue

            cross_refs = list(set(
                [f"TS {ref}" for ref in self.ts_ref_regex.findall(content)] +
                [f"clause {ref[0]}" for ref in self.clause_ref_regex.findall(content)]
            ))

            sections.append({
                "spec_number": spec_number,
                "release": release,
                "clause_path": clause_path,
                "clause_string": clause_str,
                "clause_title": clean_title,
                "level": entry["level"],
                "content": content,
                "tables": [],
                "cross_references": cross_refs,
            })

        doc.close()
        return sections

print("Parser ready.")

In [ ]:
# Cell 4: Parse ALL PDFs
import time

parser = TelecomPDFParser()
all_sections = []

for pdf_name in pdf_files:
    pdf_path = os.path.join(PDF_DIR, pdf_name)
    t0 = time.time()
    sections = parser.parse(pdf_path)
    elapsed = time.time() - t0
    print(f"  {pdf_name}: {len(sections)} sections ({elapsed:.1f}s)")
    all_sections.extend(sections)

print(f"\nTotal sections extracted: {len(all_sections)}")

# Quick stats
specs = set(s['spec_number'] for s in all_sections)
total_chars = sum(len(s['content']) for s in all_sections)
print(f"Unique specs: {len(specs)} — {sorted(specs)}")
print(f"Total content: {total_chars:,} chars (~{total_chars//4:,} tokens)")

In [ ]:
# Cell 5: Metadata Enrichment

TELECOM_CONTENT_TYPES = {
    'procedure': ['procedure', 'shall', 'initiate', 'trigger', 'upon receiving'],
    'definition': ['definition', 'means', 'refers to', 'is defined as'],
    'requirement': ['shall', 'must', 'required', 'mandatory'],
    'parameter': ['parameter', 'field', 'IE', 'information element'],
    'reference': ['reference', 'see clause', 'see TS', 'as defined in'],
}

def enrich_section(section):
    """Add content_type classification."""
    content_lower = section.get('content', '').lower()
    detected = []
    for ctype, keywords in TELECOM_CONTENT_TYPES.items():
        if any(kw in content_lower for kw in keywords):
            detected.append(ctype)
    section['content_type'] = detected[0] if detected else 'general'
    return section

all_sections = [enrich_section(s) for s in all_sections]
print(f"Enriched {len(all_sections)} sections.")

In [ ]:
# Cell 6: Hierarchical Chunker

class HierarchicalChunker:
    def __init__(self, leaf_min=200, leaf_max=500):
        self.leaf_min = leaf_min
        self.leaf_max = leaf_max

    def _approx_tokens(self, text):
        return len(text.split())

    def _split_into_sentences(self, text):
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s for s in sentences if s.strip()]

    def chunk_document(self, sections):
        all_chunks = []
        section_chunks_map = {}

        for section in sections:
            content = section.get("content", "")
            tokens = self._approx_tokens(content)
            spec_number = section['spec_number']
            clause_path = tuple(section.get("clause_path", []))
            map_key = (spec_number, clause_path)
            chunk_id = f"{spec_number}_{section['clause_string']}"

            section_chunks_map[map_key] = {
                "chunk_id": f"SEC_{chunk_id}",
                "chunk_tier": "section",
                "spec_number": spec_number,
                "clause_path": section["clause_path"],
                "clause_title": section["clause_title"],
                "content": content,
                "token_count": tokens,
                "child_ids": [],
                "metadata": section
            }

        for section in sections:
            content = section.get("content", "")
            spec_number = section['spec_number']
            clause_path = tuple(section.get("clause_path", []))
            map_key = (spec_number, clause_path)
            sec_chunk = section_chunks_map.get(map_key)
            if not sec_chunk:
                continue

            sentences = self._split_into_sentences(content)
            current_leaf_content = []
            current_leaf_tokens = 0
            leaf_idx = 0

            for sentence in sentences:
                sent_tokens = self._approx_tokens(sentence)
                if current_leaf_tokens + sent_tokens > self.leaf_max and current_leaf_content:
                    leaf_text = " ".join(current_leaf_content)
                    leaf_id = f"LEAF_{sec_chunk['chunk_id']}_{leaf_idx}"
                    all_chunks.append({
                        "chunk_id": leaf_id,
                        "chunk_tier": "leaf",
                        "parent_id": sec_chunk["chunk_id"],
                        "content": leaf_text,
                        "token_count": self._approx_tokens(leaf_text),
                        "metadata": section
                    })
                    sec_chunk["child_ids"].append(leaf_id)
                    current_leaf_content = [sentence]
                    current_leaf_tokens = sent_tokens
                    leaf_idx += 1
                else:
                    current_leaf_content.append(sentence)
                    current_leaf_tokens += sent_tokens

            if current_leaf_content:
                leaf_text = " ".join(current_leaf_content)
                leaf_id = f"LEAF_{sec_chunk['chunk_id']}_{leaf_idx}"
                all_chunks.append({
                    "chunk_id": leaf_id,
                    "chunk_tier": "leaf",
                    "parent_id": sec_chunk["chunk_id"],
                    "content": leaf_text,
                    "token_count": self._approx_tokens(leaf_text),
                    "metadata": section
                })
                sec_chunk["child_ids"].append(leaf_id)

        # Merge short trailing leaves
        merged = []
        for chunk in all_chunks:
            if chunk["chunk_tier"] == "leaf" and chunk["token_count"] < self.leaf_min and merged:
                prev = None
                for c in reversed(merged):
                    if c["chunk_tier"] == "leaf" and c.get("parent_id") == chunk.get("parent_id"):
                        prev = c
                        break
                if prev:
                    prev["content"] += " " + chunk["content"]
                    prev["token_count"] = self._approx_tokens(prev["content"])
                    continue
            merged.append(chunk)
        all_chunks = merged

        # Add sibling_ids
        parent_to_leaves = {}
        for chunk in all_chunks:
            if chunk["chunk_tier"] == "leaf":
                pid = chunk.get("parent_id", "")
                parent_to_leaves.setdefault(pid, []).append(chunk["chunk_id"])
        for chunk in all_chunks:
            if chunk["chunk_tier"] == "leaf":
                pid = chunk.get("parent_id", "")
                chunk["sibling_ids"] = [c for c in parent_to_leaves.get(pid, []) if c != chunk["chunk_id"]]

        # Add section chunks
        for _, sec_chunk in section_chunks_map.items():
            all_chunks.append(sec_chunk)

        return all_chunks

chunker = HierarchicalChunker()
chunks = chunker.chunk_document(all_sections)

leaf_chunks = [c for c in chunks if c.get('chunk_tier') == 'leaf']
sec_chunks = [c for c in chunks if c.get('chunk_tier') == 'section']
tokens = [c.get('token_count', 0) for c in leaf_chunks]

print(f"Total chunks: {len(chunks)}")
print(f"  Leaf: {len(leaf_chunks)}, Section: {len(sec_chunks)}")
print(f"  Leaf tokens — min: {min(tokens)}, max: {max(tokens)}, avg: {sum(tokens)//len(tokens)}")
print(f"  In 200-500 range: {sum(1 for t in tokens if 200 <= t <= 500)} ({100*sum(1 for t in tokens if 200 <= t <= 500)//len(tokens)}%)")

In [ ]:
# Cell 7: Build Knowledge Graph
import networkx as nx
import pickle

G = nx.DiGraph()

for section in all_sections:
    node_id = f"{section['spec_number']}_{section['clause_string']}"
    G.add_node(node_id, **{
        'spec_number': section['spec_number'],
        'clause_string': section['clause_string'],
        'clause_title': section['clause_title'],
        'level': section['level'],
        'content_type': section.get('content_type', 'general'),
    })

    # Parent-child edges based on clause_path
    clause_path = section.get('clause_path', [])
    if len(clause_path) > 1:
        parent_clause = '.'.join(clause_path[:-1])
        parent_id = f"{section['spec_number']}_{parent_clause}"
        if parent_id in G:
            G.add_edge(parent_id, node_id, relation='contains')

    # Cross-reference edges
    for ref in section.get('cross_references', []):
        if ref.startswith('TS '):
            G.add_edge(node_id, ref, relation='references')

print(f"Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Save KG
kg_path = os.path.join(OUTPUT_DIR, "section_graph.pkl")
with open(kg_path, 'wb') as f:
    pickle.dump(G, f)
print(f"KG saved to {kg_path}")

In [ ]:
# Cell 8: Embed ALL chunks with bge-large-en-v1.5 (GPU-accelerated!)
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

embed_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)

# Embed in batches
BATCH_SIZE = 64  # Larger batch on GPU
all_texts = [c["content"] for c in chunks]

print(f"Embedding {len(all_texts)} chunks...")
all_embeddings = []

for i in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="Embedding"):
    batch = all_texts[i:i+BATCH_SIZE]
    # Truncate to ~500 tokens for embedding model max_length
    batch = [t[:2000] if len(t) > 2000 else t for t in batch]
    embs = embed_model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_embeddings.extend(embs)

print(f"Done! Embedding shape: ({len(all_embeddings)}, {len(all_embeddings[0])})")

In [ ]:
# Cell 9: Save chunks with embeddings
import json

chunks_path = os.path.join(OUTPUT_DIR, "chunks_with_embeddings.jsonl")

with open(chunks_path, "w") as f:
    for chunk, emb in zip(chunks, all_embeddings):
        # Flatten metadata
        out = {
            "chunk_id": chunk["chunk_id"],
            "chunk_tier": chunk["chunk_tier"],
            "parent_id": chunk.get("parent_id", ""),
            "content": chunk["content"],
            "token_count": chunk["token_count"],
            "sibling_ids": chunk.get("sibling_ids", []),
            "child_ids": chunk.get("child_ids", []),
        }
        # Add metadata fields
        meta = chunk.get("metadata", {})
        out["spec_number"] = meta.get("spec_number", chunk.get("spec_number", ""))
        out["clause_string"] = meta.get("clause_string", chunk.get("clause_string", ""))
        out["clause_title"] = meta.get("clause_title", chunk.get("clause_title", ""))
        out["clause_path"] = meta.get("clause_path", chunk.get("clause_path", []))
        out["cross_references"] = meta.get("cross_references", chunk.get("cross_references", []))
        out["content_type"] = meta.get("content_type", "general")
        out["release"] = meta.get("release", "18")
        # Embedding as list (for JSON serialization)
        out["embedding"] = emb.tolist()
        f.write(json.dumps(out, default=str) + "\n")

# Also save without embeddings (smaller file for reference)
chunks_lite_path = os.path.join(OUTPUT_DIR, "chunks.jsonl")
with open(chunks_lite_path, "w") as f:
    for chunk in chunks:
        out = {
            "chunk_id": chunk["chunk_id"],
            "chunk_tier": chunk["chunk_tier"],
            "parent_id": chunk.get("parent_id", ""),
            "content": chunk["content"],
            "token_count": chunk["token_count"],
            "sibling_ids": chunk.get("sibling_ids", []),
            "child_ids": chunk.get("child_ids", []),
        }
        meta = chunk.get("metadata", {})
        out["spec_number"] = meta.get("spec_number", chunk.get("spec_number", ""))
        out["clause_string"] = meta.get("clause_string", chunk.get("clause_string", ""))
        out["clause_title"] = meta.get("clause_title", chunk.get("clause_title", ""))
        out["clause_path"] = meta.get("clause_path", chunk.get("clause_path", []))
        out["cross_references"] = meta.get("cross_references", chunk.get("cross_references", []))
        out["content_type"] = meta.get("content_type", "general")
        out["release"] = meta.get("release", "18")
        f.write(json.dumps(out, default=str) + "\n")

# Save embeddings separately as numpy for fast loading
emb_path = os.path.join(OUTPUT_DIR, "embeddings.npy")
np.save(emb_path, np.array(all_embeddings))

print(f"\n=== SAVED ===")
print(f"  Chunks + embeddings: {chunks_path}")
print(f"  Chunks (no embed):   {chunks_lite_path}")
print(f"  Embeddings numpy:    {emb_path}")
print(f"  Knowledge graph:     {kg_path}")

# File sizes
for fpath in [chunks_path, chunks_lite_path, emb_path, kg_path]:
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {os.path.basename(fpath)}: {size_mb:.1f} MB")

In [ ]:
# Cell 10: Final Summary
print("=" * 60)
print("INGESTION COMPLETE")
print("=" * 60)
print(f"PDFs parsed:         {len(pdf_files)}")
print(f"Sections extracted:  {len(all_sections)}")
print(f"Total chunks:        {len(chunks)}")
print(f"  Leaf chunks:       {len(leaf_chunks)}")
print(f"  Section chunks:    {len(sec_chunks)}")
print(f"KG nodes:            {G.number_of_nodes()}")
print(f"KG edges:            {G.number_of_edges()}")
print(f"Embedding dim:       {len(all_embeddings[0])}")
print("=" * 60)
print("\nDownload these files from /kaggle/working/ingestion_output/:")
print("  1. chunks_with_embeddings.jsonl  (for Qdrant indexing)")
print("  2. chunks.jsonl                  (lightweight reference)")
print("  3. embeddings.npy                (fast numpy loading)")
print("  4. section_graph.pkl             (knowledge graph)")
print("\nThen run the local indexer to load into Qdrant.")